In [3]:
import pandas as pd
import sys
import numpy as np
from data_profiling import ProfileReport
import os


In [5]:
#Profiling data


output_dir = r"/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/interim"

# Load your dataset – adjust the path if raw.csv is not in the current directory
# For example, if it's in the same folder as the script:
df = pd.read_csv("/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/raw/raw.csv")

# Alternatively, if the CSV is inside the output_dir:
# df = pd.read_csv(os.path.join(output_dir, "raw.csv"))

# 2. Generate the data profiling report
profile = ProfileReport(df, title="Dataset Profiling Report", explorative=True)

# 3. Save the report as an HTML file
report_path = os.path.join(output_dir, "data_profile_report.json")
profile.to_file(report_path)

print("Data profiling report generated and saved as 'data_profile_report.json'.")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 18.13it/s]


Render JSON:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Data profiling report generated and saved as 'data_profile_report.json'.


In [7]:
#Cleaning data
# 1. Load
df = pd.read_csv(
    "/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/raw/raw.csv",
    parse_dates=["date"],
    dtype={
        "base": "category",
        "quote": "category",
        "rate": "float64",
    },
)

# 2. Inspect
print(df.info())
print(df.head())
print(df.isna().sum())
print("Duplicate rows:", df.duplicated(subset=["date", "base", "quote"]).sum())
print(df["base"].value_counts())
print(df["quote"].value_counts())

# 3. Normalize text columns
df["base"] = df["base"].astype(str).str.strip().str.upper()
df["quote"] = df["quote"].astype(str).str.strip().str.upper()

# 4. Force correct types
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["rate"] = pd.to_numeric(df["rate"], errors="coerce")

#cleaning the rate column to remove commas and convert to numeric
df["rate"] = (
    df["rate"]
    .astype(str)
    .str.replace(",", "", regex=False)
)
df["rate"] = pd.to_numeric(df["rate"], errors="coerce")

# 5. Drop rows that cannot be used
df = df.dropna(subset=["date", "base", "quote", "rate"])

# Optional: exchange rates should be positive
df = df[df["rate"] > 0]

# 6. Check duplicates before dropping
dups = df[df.duplicated(subset=["date", "base", "quote"], keep=False)]
print("Conflicting duplicates:")
print(dups.sort_values(["date", "base", "quote"]))

# If duplicates are harmless, keep last
df = df.drop_duplicates(subset=["date", "base", "quote"], keep="last")

# 7. Sort
df = df.sort_values(["base", "quote", "date"]).reset_index(drop=True)

print(df.head())
print(df.dtypes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25005 entries, 0 to 25004
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    25005 non-null  datetime64[ns]
 1   base    25005 non-null  category      
 2   quote   25005 non-null  category      
 3   rate    25005 non-null  float64       
dtypes: category(2), datetime64[ns](1), float64(1)
memory usage: 439.7 KB
None
        date base quote     rate
0 2013-01-01  SAR   AED  0.97933
1 2013-01-01  SAR   CNY  1.66280
2 2013-01-01  SAR   EUR  0.20207
3 2013-01-01  SAR   GBP  0.16508
4 2013-01-01  SAR   USD  0.26667
date     0
base     0
quote    0
rate     0
dtype: int64
Duplicate rows: 0
base
SAR    25005
Name: count, dtype: int64
quote
AED    5001
CNY    5001
EUR    5001
GBP    5001
USD    5001
Name: count, dtype: int64
Conflicting duplicates:
Empty DataFrame
Columns: [date, base, quote, rate]
Index: []
        date base quote     rate
0 2013-01-01  SAR   AE

In [8]:
df.to_csv("/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/interim/clean.csv", index=False)

In [9]:
print(df.dtypes)
print(df.columns.tolist())
print(df.isna().sum())
print(df.duplicated(subset=["date", "base", "quote"]).sum())
print(df["base"].unique())
print(df["quote"].unique())
print(df["rate"].describe())

date     datetime64[ns]
base             object
quote            object
rate            float64
dtype: object
['date', 'base', 'quote', 'rate']
date     0
base     0
quote    0
rate     0
dtype: int64
0
['SAR']
['AED' 'CNY' 'EUR' 'GBP' 'USD']
count    25005.000000
mean         0.692750
std          0.621361
min          0.155610
25%          0.226170
50%          0.266670
75%          0.979330
max          1.958400
Name: rate, dtype: float64
